# notebook_dna_load

Loads scraped Ancestry DNA match CSVs from Unity Catalog Volume into `silver_dna_match`.

## Workflow
1. Export Ancestry match pages as HTML (Chrome, Android)
2. Parse HTML → CSV using ChatGPT prompt (see ChatGPT Prompts tab in DNA_Research_Log)
3. Save CSV to `/Volumes/workspace/staging_google_drive/DNA_matches/`
4. Run this notebook — set `kit_id` widget to match the file

## Upsert logic
- New matches → INSERT
- Existing matches → UPDATE `shared_cm`, `predicted_relationship`, `side`, `has_common_ancestor_hint`,
  `tree_info`, `tree_people_count`, `managed_by`, `last_seen_date`
- Matches previously seen but absent from current file → `active = FALSE`

Match identity key: `kit_id` + `match_name`

## Kit IDs
| kit_id | Source column value | Kit owner |
|---|---|---|
| `ed_ancestry` | `Ed` or blank | Edward Alexander Ball |
| `dad_ancestry` | `Dad` | Stephen Ball |
| `gillian_ancestry` | `Gillian` | Gillian Pearson |

## Source GEDCOM xrefs (for joining to silver_person_source)
| kit_id | source_xref |
|---|---|
| `ed_ancestry` | `@S1272999949@` |
| `gillian_ancestry` | `@S1275055303@` |
| `dad_ancestry` | `@S1275093887@` |


In [0]:
dbutils.widgets.removeAll()

# CSV filename within the DNA_matches volume
dbutils.widgets.text(
    "csv_filename",
    "ancestry_matches_ed.csv",
    "CSV filename (in /Volumes/workspace/staging_google_drive/DNA_matches/)"
)

# Kit ID — determines which kit this file represents.
# If Source column is present in CSV this is used for per-row kit assignment;
# this widget provides the fallback for files with no Source column.
dbutils.widgets.dropdown(
    "default_kit_id",
    "ed_ancestry",
    ["ed_ancestry", "dad_ancestry", "gillian_ancestry"],
    "Default kit ID (used when Source column is absent or blank)"
)

CSV_FILENAME   = dbutils.widgets.get("csv_filename")
DEFAULT_KIT_ID = dbutils.widgets.get("default_kit_id")
VOLUME_PATH    = "/Volumes/workspace/staging_google_drive/DNA_matches"
CSV_PATH       = f"{VOLUME_PATH}/{CSV_FILENAME}"

print(f"CSV path:        {CSV_PATH}")
print(f"Default kit ID:  {DEFAULT_KIT_ID}")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from datetime import date

# Expected columns from the ChatGPT HTML-parse prompt.
# All read as string first; cast to correct types below.
EXPECTED_COLUMNS = [
    "Match Name",
    "Shared DNA (cM only)",
    "Side",
    "Predicted Relationship",
    "Common Ancestor",
    "Tree Info",
    "Tree People Count",
    "Avatar Type",
    "Managed By",
    "Source",
]

# Source column → kit_id mapping
KIT_ID_MAP = {
    "ed":      "ed_ancestry",
    "dad":     "dad_ancestry",
    "gillian": "gillian_ancestry",
}

raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")   # read all as string, cast explicitly below
    .option("multiLine", "true")      # notes fields can contain newlines
    .option("escape", '"')
    .csv(CSV_PATH)
)

# Validate expected columns are present
missing = [c for c in EXPECTED_COLUMNS if c not in raw.columns]
if missing:
    raise ValueError(f"CSV is missing expected columns: {missing}")

today = date.today().isoformat()

# Normalise to silver_dna_match schema
incoming = (
    raw
    # Derive kit_id from Source column; fall back to widget default
    .withColumn(
        "kit_id",
        F.when(F.lower(F.col("`Source`")).isin(list(KIT_ID_MAP.keys())),
               F.lower(F.col("`Source`")))
         .otherwise(F.lit(DEFAULT_KIT_ID))
    )
    # Map Ed/Dad/Gillian strings to kit_id values
    .replace(KIT_ID_MAP, subset=["kit_id"])
    .withColumnRenamed("`Match Name`",          "match_name")
    .withColumn("shared_cm",
        F.col("`Shared DNA (cM only)`").cast("double"))
    .withColumnRenamed("`Side`",                "side")
    .withColumnRenamed("`Predicted Relationship`", "predicted_relationship")
    .withColumn("has_common_ancestor_hint",
        F.when(F.upper(F.col("`Common Ancestor`")) == "YES", True).otherwise(False))
    .withColumnRenamed("`Tree Info`",           "tree_info")
    .withColumn("tree_people_count",
        F.col("`Tree People Count`").cast("int"))
    .withColumnRenamed("`Avatar Type`",         "avatar_type")
    .withColumnRenamed("`Managed By`",          "managed_by")
    .withColumn("last_seen_date", F.lit(today))
    .withColumn("active",        F.lit(True))
    .select(
        "kit_id",
        "match_name",
        "shared_cm",
        "side",
        "predicted_relationship",
        "has_common_ancestor_hint",
        "tree_info",
        "tree_people_count",
        "avatar_type",
        "managed_by",
        "last_seen_date",
        "active"
    )
    # Drop rows with no match name (blank rows at end of CSV)
    .where(F.col("match_name").isNotNull() & (F.col("match_name") != ""))
)

print(f"Incoming rows: {incoming.count()}")
incoming.show(5, truncate=False)


In [0]:
# Create silver_dna_match if this is the first run.
# On subsequent runs the MERGE below handles everything.
spark.sql("""
    CREATE TABLE IF NOT EXISTS genealogy.silver_dna_match (
        kit_id                   STRING    COMMENT 'e.g. ed_ancestry, dad_ancestry, gillian_ancestry',
        match_name               STRING    COMMENT 'Ancestry username or display name',
        shared_cm                DOUBLE    COMMENT 'Shared centimorgans from platform',
        side                     STRING    COMMENT 'Maternal / Paternal / Unassigned',
        predicted_relationship   STRING    COMMENT 'Ancestry predicted relationship label',
        has_common_ancestor_hint BOOLEAN   COMMENT 'True if Ancestry shows a common ancestor hint',
        tree_info                STRING    COMMENT 'No trees / Unlinked tree / Public linked tree etc.',
        tree_people_count        INT       COMMENT 'Number of people in match tree if available',
        avatar_type              STRING    COMMENT 'Male / Female / Photo',
        managed_by               STRING    COMMENT 'Manager name if kit is managed by someone else',
        first_seen_date          STRING    COMMENT 'Date first loaded into this table (ISO 8601)',
        last_seen_date           STRING    COMMENT 'Date last present in a scrape for this kit (ISO 8601)',
        active                   BOOLEAN   COMMENT 'False if match no longer appears in scrape exports'
    )
    USING DELTA
    COMMENT 'Raw scraped DNA match lists. One row per kit + match_name. Upserted on each CSV load.'
""")
print("Table ready")


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "genealogy.silver_dna_match")

# MERGE on kit_id + match_name.
# Matched rows: update mutable fields + last_seen_date, keep first_seen_date.
# New rows: insert with first_seen_date = last_seen_date = today.
(
    target.alias("t")
    .merge(
        incoming.alias("s"),
        "t.kit_id = s.kit_id AND t.match_name = s.match_name"
    )
    .whenMatchedUpdate(set={
        "shared_cm":                "s.shared_cm",
        "side":                     "s.side",
        "predicted_relationship":   "s.predicted_relationship",
        "has_common_ancestor_hint": "s.has_common_ancestor_hint",
        "tree_info":                "s.tree_info",
        "tree_people_count":        "s.tree_people_count",
        "avatar_type":              "s.avatar_type",
        "managed_by":               "s.managed_by",
        "last_seen_date":           "s.last_seen_date",
        "active":                   "true",   # reactivate if previously deactivated
    })
    .whenNotMatchedInsert(values={
        "kit_id":                   "s.kit_id",
        "match_name":               "s.match_name",
        "shared_cm":                "s.shared_cm",
        "side":                     "s.side",
        "predicted_relationship":   "s.predicted_relationship",
        "has_common_ancestor_hint": "s.has_common_ancestor_hint",
        "tree_info":                "s.tree_info",
        "tree_people_count":        "s.tree_people_count",
        "avatar_type":              "s.avatar_type",
        "managed_by":               "s.managed_by",
        "first_seen_date":          "s.last_seen_date",
        "last_seen_date":           "s.last_seen_date",
        "active":                   "true",
    })
    .execute()
)

print("Merge complete")


In [0]:
# Any match for this kit that was NOT in the current CSV gets marked inactive.
# We scope this only to the kit(s) present in the incoming file to avoid
# accidentally deactivating matches from other kits.
#
# Logic: active=TRUE rows for this kit whose match_name is not in incoming.

kits_in_file = [row["kit_id"] for row in incoming.select("kit_id").distinct().collect()]
print(f"Kits in this file: {kits_in_file}")

incoming_names = incoming.select("kit_id", "match_name")

(
    target.alias("t")
    .merge(
        incoming_names.alias("s"),
        "t.kit_id = s.kit_id AND t.match_name = s.match_name"
    )
    .whenNotMatchedBySourceAnd(
        # Only deactivate rows for kits that were in this file
        f"t.active = true AND t.kit_id IN ({','.join(repr(k) for k in kits_in_file)})"
    )
    .updateSet({"active": "false"})
    .execute()
)

print("Inactive flag update complete")


In [0]:
summary = (
    spark.table("genealogy.silver_dna_match")
    .groupBy("kit_id", "active")
    .count()
    .orderBy("kit_id", "active")
)
print("silver_dna_match row counts:")
summary.show()
